# 3.4 Tiling 设计原则：图纸为什么这样画

Kernel 要多核并行、要分块计算，就必须有人告诉它：**切多大的块、切到哪些核、每个核循环多少次**。这些参数在 Host 侧由 **TilingFunc** 一次性算好——这就是 Tiling。

![Tiling 流程](images/tiling_flow.svg)

## 1. 为什么切分参数要在 Host 侧算？

Kernel 跑在 NPU 的 AI Core 上，其中 **Scalar 单元负责流程控制**，它的算力极其有限。如果把「算基本块、算偏移、查表分核」都放到 Kernel 里：

1. **Scalar 资源宝贵**——每个核都要重复算一遍相同的切分逻辑，白白消耗控制流算力；
2. **TilingData 要尽量小**——TilingData 会随任务下发到每个核，体积大意味着更大的图开销与同地址访问开销。

所以原则是：**能在 Tiling 侧算好的，绝不下放到 Kernel**。Kernel 拿到的 TilingData 是一份「填好的表格」，照表执行即可。

## 2. 五大设计原则

以 FA 算子的真实 Tiling 实现为参照，设计原则可归纳为五条：

### 原则一：基本块要在交互与效率之间找平衡

AIC 与 AIV 的算力不同、L2 Cache 大小有限、两者交互有通信开销。基本块的选择（需要反复测试找最优解）要同时满足：

- **减少交互次数**，保证单次交互的效率；
- 保证 **Cube 的计算与访存效率**、**Vector 的计算效率**。

### 原则二：用 TilingKey 做编译期隔离，解决 ICache Miss

融合算子是所有算子里计算逻辑最复杂的（多了很多通信与可选输入），众多输入组合导致**代码段膨胀**、Scalar 计算量增加。而指令缓存 ICache 的容量是有限的（AIC 32KB / AIV 16KB）——全部场景的代码混在一起，运行时大量指令缓存未命中（Cache Miss），处理器被迫去慢存储取指令。

解法：**用 TilingKey 把不同场景的代码在编译期就隔离开**，每个 Kernel 只包含自己场景的代码段。

![TilingKey 分发](images/tilingkey_dispatch.svg)

> Cache Miss 是计算机系统最关键的性能瓶颈之一：CPU 需要的数据不在缓存时，必须从更慢的层级获取，造成巨大性能损失。

### 原则三：Kernel 侧 Scalar 省着用

把尽可能多的计算提取到 Tiling 侧（在 CPU 上算，白嫖 x86 的算力），同时控制 TilingData 体积。

### 原则四：模板之间共用公共代码

不同 Tiling 模板之间，关于**输入的解析、平台信息的解析、负载均衡**的代码是共用的，避免每个模板都复制一份。

### 原则五：负载均衡算法不宜过复杂

过于复杂的负载均衡往往不会带来太好收益，反而增加 Tiling 自身的耗时——Tiling 是每次算子调用都要跑的，慢了直接影响端到端。

## 3. 模板与优先级调度

FA 有多个 Tiling 模板（适配不同输入范围与排布）。框架按**注册优先级**依次调用每个模板的 `DoTiling`，靠返回值驱动调度：

```cpp
// Tiling 执行框架（简化）
ge::graphStatus DoTiling()
{   // 按照注册优先级调用每一个 DoTiling 函数
    // 如果高优先级的模板返回不予支持，那么继续调用较低优先级模板
    auto ret = GetShapeAttrsInfo();   // 1. 取输入 shape、属性
    if (ret != ge::GRAPH_SUCCESS) return ret;
    ret = GetPlatformInfo();           // 2. 取硬件平台信息
    if (ret != ge::GRAPH_SUCCESS) return ret;
    if (!IsCapable())                  // 3. 判断本模板是否支持当前场景
        return ge::GRAPH_PARAM_INVALID;
    ret = DoOpTiling();                // 4. 基本块选择、核间切分、负载均衡
    if (ret != ge::GRAPH_SUCCESS) return ret;
    ret = DoLibApiTiling();            // 5. 核内 AscendC 高层 API tiling
    if (ret != ge::GRAPH_SUCCESS) return ret;
    ret = GetWorkspaceSize();          // 6. 计算 workspace 大小
    if (ret != ge::GRAPH_SUCCESS) return ret;
    ret = PostTiling();                // 7. 后处理（tilingData 大小、dropout 配置等）
    if (ret != ge::GRAPH_SUCCESS) return ret;
    context_->SetTilingKey(0);         // 8. 设置 TilingKey
    return ge::GRAPH_SUCCESS;
}
```

三个返回值的语义：

| 返回值 | 含义 | 框架行为 |
|--|--|--|
| `GRAPH_SUCCESS` | 成功且无需再往下执行 | 采纳本模板，流程结束 |
| `GRAPH_FAILED` | 失败 | 终止整个 Tiling 流程，报错 |
| `GRAPH_PARAM_INVALID` | **本模板不支持该场景** | **继续尝试下一个（优先级更低的）模板** |

> 这个「三态返回 + 优先级链」是融合算子多模板管理的通用模式。

## 4. 按谁切？Cube/Vector 的切分权衡

分离架构下 AIC 算矩阵乘、AIV 算逐元素逻辑。FA 的计算量账单（基本块视角）：

| 单元 | 计算量 |
|--|--|
| Cube | S1 × S2 × D × 2（两次矩阵乘 bmm1、bmm2） |
| Vector | S1 × (S2 + D)（softmax/mask/scale + update） |

基本块要「按谁切」有经典权衡：

- **按 Cube 切**：两个相关的 Vector 任务落在不同核，一次运算要处理两次消息，**通信开销增加**；
- **按 Vector 切**：Cube 变成「委托人」，两个 Vector 核各自的 Cube 请求没有关联性，会**重复搬运右矩阵**。

实际实现（见 3.5 节）选择的是：以 B、N2、S1 为核间切分轴、通过 `nRatio` 配比把 Cube 块与 Vector 块绑成一个组合——这正呼应了第 1 章「AIC+AIV 组合核」的启动方式。

## 5. 本节小结

- Tiling 存在的根本原因：**Kernel 侧 Scalar 太宝贵**，切分逻辑上移到 Host。
- 五大原则：基本块权衡、TilingKey 编译期隔离（治 ICache Miss）、Scalar 上提、模板共用代码、负载均衡从简。
- 多模板按优先级依次尝试，`GRAPH_PARAM_INVALID` 表示「我不支持，请试下一个」。
- Cube/Vector 切分各有代价，FA 用组合核 + nRatio 配比来绑定两侧任务。

## 小测验

1. Tiling 有哪两大「不能太重」的自相矛盾约束？（提示：Kernel 侧资源 vs 每次 Tiling 的执行成本）
2. TilingKey 解决什么问题？如果不用它，不同场景的代码混编会付出什么代价？
3. 模板返回 `GRAPH_PARAM_INVALID` 与返回 `GRAPH_FAILED`，框架分别怎么处理？
4. 为什么「把切分逻辑全部写进 Kernel、每个核自己算」不是好设计？